# Junk Property Classification — End-to-End Data Science Project

**Objective:** classify properties as **Junk (1)** vs **Non-Junk (0)** using listing, pricing, property, geographic and market features.

This notebook is designed around `Property_train.csv` and extends the supplied notebook with:
- reproducible preprocessing
- date feature engineering
- leakage-safe sklearn pipelines
- imbalanced-class evaluation
- threshold optimization
- model persistence
- explainable feature-importance output


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier

DATA_PATH = Path("Property_train.csv")
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## 1. Target and class balance

In [ ]:
print(df["Junk"].value_counts())
print(df["Junk"].value_counts(normalize=True).rename("share"))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Junk")
plt.title("Target Class Distribution")
plt.tight_layout()
plt.show()


## 2. Data quality

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique()
}).sort_values(["missing", "unique"], ascending=False)
quality


## 3. Feature engineering

In [ ]:
data = df.copy()

data["ListDate"] = pd.to_datetime(data["ListDate"], errors="coerce")
data["ListYear"] = data["ListDate"].dt.year
data["ListMonth"] = data["ListDate"].dt.month
data["ListDay"] = data["ListDate"].dt.day
data["ListDayOfWeek"] = data["ListDate"].dt.dayofweek
data["ListQuarter"] = data["ListDate"].dt.quarter
data.drop(columns=["ListDate"], inplace=True)

price_cols = [f"PriceIndex{i}" for i in range(1, 10)]
for col in price_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

X = data.drop(columns="Junk")
y = data["Junk"].astype(int)

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))


## 4. Leakage-safe preprocessing and XGBoost

In [ ]:
numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True))
])

preprocess = ColumnTransformer([
    ("numeric", numeric, num_cols),
    ("categorical", categorical, cat_cols)
])

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)
prob = pipe.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, prob), 4))


## 5. Threshold optimization

In [ ]:
rows = []
for threshold in np.arange(0.05, 0.56, 0.01):
    pred = (prob >= threshold).astype(int)
    rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })

thresholds = pd.DataFrame(rows)
thresholds.sort_values("f1", ascending=False).head(10)


In [ ]:
best = thresholds.loc[thresholds["f1"].idxmax()]
threshold = float(best["threshold"])
pred = (prob >= threshold).astype(int)

print("Selected threshold:", threshold)
print(classification_report(y_test, pred, digits=4, zero_division=0))

cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Non-Junk", "Junk"],
            yticklabels=["Non-Junk", "Junk"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — threshold={threshold:.2f}")
plt.tight_layout()
plt.show()


## 6. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
precision, recall, _ = precision_recall_curve(y_test, prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, prob):.3f}")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

plt.figure(figsize=(7,5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve — AP={average_precision_score(y_test, prob):.3f}")
plt.show()


## 7. Feature importance

In [ ]:
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
importance = pipe.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({"feature": feature_names, "importance": importance})
    .sort_values("importance", ascending=False)
)

feature_importance.head(25)


In [ ]:
plt.figure(figsize=(9,8))
top = feature_importance.head(20).sort_values("importance")
plt.barh(top["feature"], top["importance"])
plt.xlabel("XGBoost importance")
plt.title("Top 20 Model Features")
plt.tight_layout()
plt.show()


## 8. Save the model

In [ ]:
bundle = {
    "pipeline": pipe,
    "threshold": threshold,
    "feature_columns": X.columns.tolist()
}
joblib.dump(bundle, "junk_property_xgb.joblib")
print("Saved junk_property_xgb.joblib")


## 9. Business interpretation

The model produces a **Junk probability** rather than only a hard label. The threshold can be adjusted according to the cost of false negatives versus false positives.

For a screening workflow:
- **High recall:** useful when missing a Junk property is expensive.
- **High precision:** useful when manual review capacity is limited.
- **F1:** balances precision and recall for a single operating point.

The current dataset is imbalanced, so accuracy alone should not be used as the primary model-selection criterion.
